In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import sqlite3

In [2]:
import pipeline
pipeline.run()

c:\Users\sokso\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'acquired null rate': 53.012940096532525, 'relinquished null rate': 46.98705990346747}
Pipeline complete.


## Row counts per table

In [3]:
conn = sqlite3.connect('../database/nba_predictor.db')

tables = ['players', 'player_stats', 'players_injuries', 'player_season_features']
for table in tables:
    count = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table}: {count:,} rows')

players: 2,377 rows
player_stats: 11,659 rows
players_injuries: 52,248 rows
player_season_features: 11,659 rows


## Referential integrity checks

In [4]:
stats_join = pd.read_sql_query("""
    SELECT
        players.player_id,
        players.first_name,
        players.last_name
    FROM players
    INNER JOIN player_stats
    ON players.player_id = player_stats.player_id
""", conn)

print(f'player_stats rows with a matching player: {len(stats_join):,}')
stats_join.head()

player_stats rows with a matching player: 11,659


,player_id,first_name,last_name
0,3,Grant,Long
1,3,Grant,Long
2,3,Grant,Long
3,15,Eric,Piatkowski
4,15,Eric,Piatkowski


In [5]:
injuries_join = pd.read_sql_query("""
    SELECT
        players.player_id,
        players.first_name,
        players.last_name
    FROM players
    INNER JOIN players_injuries
    ON players.player_id = players_injuries.player_id
""", conn)

print(f'players_injuries rows with a matching player: {len(injuries_join):,}')
injuries_join.head()

players_injuries rows with a matching player: 52,248


,player_id,first_name,last_name
0,201166,Aaron,Brooks
1,201166,Aaron,Brooks
2,201166,Aaron,Brooks
3,201166,Aaron,Brooks
4,201166,Aaron,Brooks


## Data quality profile

In [6]:
stats_df = pd.read_sql_query('SELECT * FROM player_stats', conn)

null_report = stats_df.isna().mean().mul(100).round(2).to_frame('null_%')
null_report['null_count'] = stats_df.isna().sum()
print('player_stats null profile:')
null_report

player_stats null profile:


,null_%,null_count
points_per_game,0.0,0
assists_per_game,0.0,0
rebounds_per_game,0.0,0
blocks_per_game,0.0,0
steals_per_game,0.0,0
minutes_per_game,0.0,0
field_goal_percentage,0.0,0
games_played,0.0,0
team,0.0,0
position,0.0,0


In [7]:
injuries_df = pd.read_sql_query('SELECT * FROM players_injuries', conn)

null_report_inj = injuries_df.isna().mean().mul(100).round(2).to_frame('null_%')
null_report_inj['null_count'] = injuries_df.isna().sum()
print('players_injuries null profile:')
null_report_inj

players_injuries null profile:


,null_%,null_count
injury_id,0.0,0
player_id,0.0,0
season,0.0,0
date_of_injury,0.0,0
injury_type,0.0,0
required_surgery,0.0,0
games_missed,0.0,0


In [8]:
stats_dupes = stats_df.duplicated(subset=['player_id', 'season']).sum()
injuries_dupes = injuries_df.duplicated(subset=['player_id', 'season', 'date_of_injury']).sum()

print(f'player_stats duplicate (player_id, season) pairs: {stats_dupes}')
print(f'players_injuries duplicate rows: {injuries_dupes}')

player_stats duplicate (player_id, season) pairs: 0
players_injuries duplicate rows: 34882


In [9]:
seasons = pd.read_sql_query('SELECT DISTINCT season FROM player_stats ORDER BY season', conn)

print(f'Seasons covered: {seasons.iloc[0, 0]} to {seasons.iloc[-1, 0]}')
print(f'Total seasons: {len(seasons)}')
print(seasons['season'].tolist())

Seasons covered: 2000-01 to 2023-24
Total seasons: 24
['2000-01', '2001-02', '2002-03', '2003-04', '2004-05', '2005-06', '2006-07', '2007-08', '2008-09', '2009-10', '2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24']


In [10]:
conn.close()
print('Done.')

Done.
